In [1]:
import torch
print(torch.cuda.is_available())
print(torch.cuda.get_device_name(0) if torch.cuda.is_available() else "no gpu")

True
Tesla T4


In [2]:
!pip install ultralytics gdown -q

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.5/45.5 kB 2.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 24.8 MB/s eta 0:00:00


In [3]:
import gdown
DRIVE_FILE_ID = "1AD-DW45gqZpj1Jt3jj7jMXxhf3MK83dS"
gdown.download(id=DRIVE_FILE_ID, output="/kaggle/working/dataset.zip", quiet=False)

Downloading...
From (original): https://drive.google.com/uc?id=1AD-DW45gqZpj1Jt3jj7jMXxhf3MK83dS
From (redirected): https://drive.google.com/uc?id=1AD-DW45gqZpj1Jt3jj7jMXxhf3MK83dS&confirm=t&uuid=00d5efab-24d4-4afd-b993-925a24bbe326
To: /kaggle/working/dataset.zip
100%|██████████| 1.47G/1.47G [00:22<00:00, 66.9MB/s]


'/kaggle/working/dataset.zip'

In [4]:
import zipfile, os
LOCAL_PATH = "/kaggle/working/data_extracted"
os.makedirs(LOCAL_PATH, exist_ok=True)
with zipfile.ZipFile("/kaggle/working/dataset.zip", "r") as z:
    z.extractall(LOCAL_PATH)
print(os.listdir(LOCAL_PATH))

['data.yaml', 'data']


In [5]:
train_path = f"{LOCAL_PATH}/data/raw/images/train"
val_path = f"{LOCAL_PATH}/data/raw/images/val"
with open("/kaggle/working/data_kaggle.yaml", "w") as f:
    f.write(f"train: {train_path}\nval: {val_path}\nnc: 1\nnames: [\"person\"]\n")
print(os.path.exists(train_path), os.path.exists(val_path))

True True


In [6]:
from ultralytics import YOLO
model = YOLO("yolo11s.pt")

Creating new Ultralytics Settings v0.0.7 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.


In [7]:
results = model.train(
    data="/kaggle/working/data_kaggle.yaml",
    epochs=100,
    patience=40,
    optimizer="AdamW",
    lr0=0.001,
    imgsz=640,
    batch=16,
    project="/kaggle/working/runs",
    name="person_detector_v11s",
    save_period=10,
    plots=True,
)

Ultralytics 8.4.115 🚀 Python-3.12.13 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14912MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=False, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/kaggle/working/data_kaggle.yaml, degrees=0.0, deterministic=True, device=, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=100, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.001, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolo11s.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=person_detector_v11s, nbs=6

In [8]:
import glob
matches = glob.glob("/kaggle/working/runs/**/best.pt", recursive=True)
best_weights_path = max(matches, key=os.path.getmtime)

from ultralytics import YOLO
best_model = YOLO(best_weights_path)
metrics = best_model.val(data="/kaggle/working/data_kaggle.yaml", imgsz=640, plots=True)

print("mAP50:", metrics.box.map50)
print("mAP50-95:", metrics.box.map)
print("precision:", metrics.box.mp)
print("recall:", metrics.box.mr)

Ultralytics 8.4.115 🚀 Python-3.12.13 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14912MiB)
YOLO11s summary (fused): 101 layers, 9,413,187 parameters, 0 gradients, 21.4 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 2766.9±660.1 MB/s, size: 169.5 KB)
val: Scanning /kaggle/working/data_extracted/data/raw/labels/val.cache... 1000 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 1000/1000 419.4Mit/s 0.0s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 63/63 5.2it/s 12.1s
                   all       1000       4308      0.779      0.678      0.755      0.508
Speed: 0.8ms preprocess, 7.7ms inference, 0.0ms loss, 1.0ms postprocess per image
Results saved to /kaggle/working/runs/detect/val
mAP50: 0.7554445599556943
mAP50-95: 0.5075416854275738
precision: 0.7785441037044379
recall: 0.6781430432935727
